# Summarize Each Firm

| Item | Description |
|---|---|
| Goal | One summary per firm, written by the language model from the firm's yearly filing summaries, as the reduce stage of T1 writes one hearing summary from its segment summaries. One call per firm; a firm's yearly summaries always fit one call. |
| Input | `settings.json` from T3.3.1 in the T3.3 Interim folder; `filing_summaries.parquet` from T3.3.3 in the Processed folder; `filings.parquet` from T3.1.1. One run does one input set, one family, and one prompt version, set in Step 2. |
| Output | In the T3.3 Interim folder, `firm_summaries/<set>/<family>/<version>/`: `firms.jsonl` (one line per firm), `firms.calls.jsonl` (every call), and `firms.json` (the run record). |
| Model | `Qwen/Qwen3.5-4B` in bfloat16 on one GPU, the settings of T3.3.1. A firm summary over 512 encoder tokens is re-asked shorter, up to twice; one still over the limit loses trailing sentences until it fits, and the record says so. Finished firms are read back and skipped, so a run can be resumed. |
| Run | Ran in October 2026 on one NVIDIA A40 (`J - Jobs/run_T3.3.4_summarize_firms.sbatch`) for the all-years set (1,058 firms) and the three-year set (841 firms), family `summary_lines_of_business`, version `item1-mapreduce-1.3`. Not rerun, because sampling would write new texts; Step 8 reads the two run records back. |


## Step 1. Import packages


In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations, the run, and the batch settings

Change `SET` to run the other sample; the SLURM wrapper sets it from the environment.


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.3 - Firm Side - Summaries of the 10-K Item 1 Filings")
NOTEBOOK_NAME = "T3.3.4 - Production - Summarize Each Firm"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SETTINGS_FILE = INTERIM_FOLDER / "settings.json"
TABLE_FILE = PROCESSED_FOLDER / "filing_summaries.parquet"
FILINGS_FILE = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.1 - Sample - Firms and 10-K Item 1 Filings" / "Processed" / "filings.parquet"
FIRM_FOLDER = INTERIM_FOLDER / "firm_summaries"
ENCODER = "BAAI/bge-large-en-v1.5"  # its tokenizer counts the tokens of each summary against the 512 limit

SET = os.environ.get("SET", "all_years")  # "all_years" or "three_years"
FAMILY = os.environ.get("FAMILY", "summary_lines_of_business")
VERSION = os.environ.get("VERSION", "item1-mapreduce-1.3")  # the prompt version of the filing summaries to read; must equal the settings version
LIMIT = None  # smoke test: only the first N firms
MAX_BATCH_TOKENS = 48_000  # total padded prompt tokens per forward pass
MAX_BATCH_PROMPTS = 16  # prompts per forward pass at most
SEED = 0
SHORTER_TARGET_WORDS = ["150 to 200", "120 to 150"]  # re-asks for a firm summary that exceeds the encoder limit
PROGRESS_EVERY = 10
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (INTERIM_FOLDER,):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [ ]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Read the settings, the filing summaries, and the sample

Every filing of the set must have a summary at the requested version. A firm's yearly summaries are gathered in fiscal-year order; firms already in the output file are skipped.


In [ ]:
settings = json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
assert FAMILY in settings["families"] and VERSION == settings["prompt_version"], f"settings are at {settings['prompt_version']}, requested {VERSION}"
prompts = settings["prompts"]
table = pd.read_parquet(TABLE_FILE)
table = table.loc[table["family"].eq(FAMILY) & table["prompt_version"].eq(VERSION)]
filings = pd.read_parquet(FILINGS_FILE)
flag = "in_three_years" if SET == "three_years" else "in_all_years"
wanted = filings.loc[filings[flag].eq(1), ["accessionNo", "gvkey", "conm", "fyear"]]
missing = set(wanted["accessionNo"]) - set(table["accessionNo"])
assert not missing, f"{len(missing)} filings of the {SET} sample have no {FAMILY} summary at {VERSION}"
yearly = wanted.merge(table[["accessionNo", "summary"]], on="accessionNo", how="left", validate="one_to_one")
firms = yearly.groupby("gvkey", sort=True).agg(conm=("conm", "first"), n_filings=("accessionNo", "size"), years=("fyear", lambda s: sorted(int(y) for y in s)))
if LIMIT is not None:
    firms = firms.head(LIMIT)
output_folder = FIRM_FOLDER / SET / FAMILY / VERSION
output_folder.mkdir(parents=True, exist_ok=True)
output_file, calls_file = output_folder / "firms.jsonl", output_folder / "firms.calls.jsonl"
done = {json.loads(line)["gvkey"] for line in output_file.read_text(encoding="utf-8").splitlines()} if output_file.exists() else set()
firms = firms.loc[~firms.index.isin(done)]
print(f"Set {SET}, family {FAMILY}, version {VERSION}: {len(firms):,} firms to do ({len(done):,} already done), {int(firms['n_filings'].sum()):,} yearly summaries")


## Step 5. Load the language model and the encoder's tokenizer


In [ ]:
assert torch.cuda.is_available(), "run this notebook on a GPU node"
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(settings["model"])
tokenizer.padding_side = "left"
pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(settings["model"], dtype=torch.bfloat16, device_map="cuda").eval()
bge_tokenizer = AutoTokenizer.from_pretrained(ENCODER)
gpu_name = torch.cuda.get_device_name(0)
run_started = datetime.now(timezone.utc).isoformat(timespec="seconds")
calls_handle = calls_file.open("a", encoding="utf-8")
n_calls = 0
print(f"Model {settings['model']} on {gpu_name}, {next(model.parameters()).dtype}; prompt version {settings['prompt_version']}")


## Step 6. Define the firm prompt and the batched generation

The firm prompt presents the yearly summaries marked with their fiscal years and the family's per-firm task. Batching, trimming, and logging are those of T3.3.2.


In [ ]:
def render(user_prompt):
    messages = [{"role": "system", "content": prompts["system"]["reduce"]}, {"role": "user", "content": user_prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=settings["enable_thinking"])


def model_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def trim_to_sentence(text):
    text = text.rstrip()
    if not text or text.endswith((".", "!", "?", '."', ".'", ".)")):
        return text
    cut = max(text.rfind(mark) for mark in ".!?")
    if cut <= 0:
        return text
    end = cut + 1
    while end < len(text) and text[end] in "\"')]":
        end += 1
    return text[:end].rstrip()


def clean(text):
    return re.sub(r"^<think>.*?</think>\s*", "", text.strip(), flags=re.DOTALL).strip()


def firm_prompt(gvkey, target_words):
    firm = firms.loc[gvkey]
    rows = yearly.loc[yearly["gvkey"].eq(gvkey)].sort_values("fyear")
    text = "\n\n".join(f"[fiscal {int(row.fyear)}] {row.summary}" for row in rows.itertuples(index=False))
    return prompts["firm_opening"].format(company=firm["conm"], years=", ".join(str(year) for year in firm["years"]), text=text) + prompts["firm_tasks"][FAMILY].format(target_words=target_words)


def generate_batched(items, max_new_tokens):
    """Generate completions for (gvkey, user_prompt) items; returns {gvkey: text}. Logs every call."""
    global n_calls
    rendered = {key: render(user_prompt) for key, user_prompt in items}
    lengths = {key: model_tokens(text) for key, text in rendered.items()}
    order = sorted(rendered, key=lambda key: lengths[key])
    batches, current = [], []
    for key in order:
        longest = max([lengths[k] for k in current] + [lengths[key]])
        if current and (len(current) + 1 > MAX_BATCH_PROMPTS or longest * (len(current) + 1) > MAX_BATCH_TOKENS):
            batches.append(current)
            current = []
        current.append(key)
    if current:
        batches.append(current)
    results, user_prompts = {}, dict(items)
    for batch_number, batch in enumerate(batches, start=1):
        encoded = tokenizer([rendered[key] for key in batch], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        started = time.time()
        with torch.no_grad():
            output_ids = model.generate(**encoded, max_new_tokens=max_new_tokens, pad_token_id=pad_token_id, **settings["generation"])
        generated = output_ids[:, encoded["input_ids"].shape[1]:].cpu()
        del output_ids, encoded
        torch.cuda.empty_cache()
        seconds = time.time() - started
        for row, key in enumerate(batch):
            ids = generated[row]
            ids = ids[ids != pad_token_id]
            text = clean(tokenizer.decode(ids, skip_special_tokens=True))
            hit_cap = int(len(ids)) >= max_new_tokens
            if hit_cap:
                text = trim_to_sentence(text)
            results[key] = text
            calls_handle.write(json.dumps({"gvkey": key, "family": FAMILY, "stage": "firm", "prompt_version": settings["prompt_version"], "model": settings["model"],
                                           "input_sha256": hashlib.sha256(user_prompts[key].encode("utf-8")).hexdigest(), "input_tokens": lengths[key],
                                           "output_tokens": int(len(ids)), "hit_cap": hit_cap, "seconds_batch": round(seconds, 1), "batch_size": len(batch),
                                           "output": text, "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}, ensure_ascii=False) + "\n")
            n_calls += 1
        calls_handle.flush()
        if batch_number % PROGRESS_EVERY == 0 or batch_number == len(batches):
            print(f"  firm: batch {batch_number}/{len(batches)}, {len(batch)} prompts, longest {max(lengths[k] for k in batch):,} tokens, {seconds:.0f}s", flush=True)
    return results


## Step 7. One call per firm, the length rule, and the record

A summary over the encoder limit is re-asked shorter, up to twice; one still over the limit loses trailing sentences until it fits. One line per firm goes to the output file and a run record beside it.


In [ ]:
stage_started = time.time()
items = [(gvkey, firm_prompt(gvkey, settings["target_words"]["firm"])) for gvkey in firms.index]
print(f"Firm calls: {len(items):,}")
final = generate_batched(items, settings["max_new_tokens"]["reduce"]) if items else {}
rounds = {gvkey: 1 for gvkey in firms.index}
for target_words in SHORTER_TARGET_WORDS:
    too_long = [gvkey for gvkey in firms.index if len(bge_tokenizer(final[gvkey])["input_ids"]) > settings["max_bge_tokens"]]
    if not too_long:
        break
    print(f"Re-asking {len(too_long):,} firms at {target_words} words")
    redo = generate_batched([(gvkey, firm_prompt(gvkey, target_words)) for gvkey in too_long], settings["max_new_tokens"]["reduce"])
    for gvkey in too_long:
        final[gvkey] = redo[gvkey]
        rounds[gvkey] += 1
trimmed = {gvkey: False for gvkey in firms.index}
for gvkey in firms.index:
    while len(bge_tokenizer(final[gvkey])["input_ids"]) > settings["max_bge_tokens"]:
        sentences = re.split(r"(?<=[.!?])\s+", final[gvkey].strip())
        assert len(sentences) > 1, f"{gvkey}: cannot trim a one-sentence summary under the limit"
        final[gvkey] = " ".join(sentences[:-1])
        trimmed[gvkey] = True

records_written = 0
with output_file.open("a", encoding="utf-8") as handle:
    for gvkey, firm in firms.iterrows():
        text = final[gvkey]
        n_bge_tokens = len(bge_tokenizer(text)["input_ids"])
        handle.write(json.dumps({"gvkey": gvkey, "conm": firm["conm"], "set": SET, "family": FAMILY, "prompt_version": settings["prompt_version"], "n_filings": int(firm["n_filings"]),
                                 "years": firm["years"], "summary": text, "n_words": len(text.split()), "n_bge_tokens": n_bge_tokens,
                                 "within_bge_limit": n_bge_tokens <= settings["max_bge_tokens"], "reduce_rounds": rounds[gvkey], "trimmed": trimmed[gvkey], "run_started": run_started}, ensure_ascii=False) + "\n")
        records_written += 1
calls_handle.close()
seconds = time.time() - stage_started
(output_folder / "firms.json").write_text(json.dumps({
    "set": SET, "family": FAMILY, "prompt_version": settings["prompt_version"], "firms_done_this_run": records_written, "firms_skipped_as_done": len(done),
    "firms_trimmed": sum(trimmed.values()), "model_calls": n_calls, "model": settings["model"], "gpu": gpu_name, "generation": settings["generation"],
    "enable_thinking": settings["enable_thinking"], "max_new_tokens": settings["max_new_tokens"]["reduce"], "seed": SEED, "torch": torch.__version__,
    "seconds": round(seconds, 1), "run_started": run_started, "finished": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=2), encoding="utf-8")
print(f"Wrote {records_written:,} firm summaries ({n_calls:,} model calls) to {output_file.name} in {seconds / 60:.1f} minutes")


## Step 8. Read the run records back

Evidence of the October runs, one row per set. The all-years run was resumed once for a single firm, which is why its last record shows one firm done and 1,057 skipped; the call log holds every call of both sessions. This cell runs without the model.


In [3]:
rows = []
for record_file in sorted(FIRM_FOLDER.glob("*/*/*/firms.json")):
    record = json.loads(record_file.read_text(encoding="utf-8"))
    calls = pd.read_json(record_file.with_name("firms.calls.jsonl"), lines=True)
    firms_written = pd.read_json(record_file.with_name("firms.jsonl"), lines=True)
    rows.append({"set": record["set"], "family": record["family"], "prompt_version": record["prompt_version"], "firms": firms_written["gvkey"].nunique(), "calls_logged": len(calls),
                 "re_asked_firms": int(firms_written["reduce_rounds"].gt(1).sum()), "trimmed_firms": int(firms_written.get("trimmed", pd.Series(False, index=firms_written.index)).fillna(False).astype(bool).sum()),
                 "gpu": record["gpu"], "gpu_minutes_logged": round(calls.drop_duplicates(["at", "batch_size"])["seconds_batch"].sum() / 60, 1), "last_session_finished": record["finished"][:10]})
display(pd.DataFrame(rows))


,set,family,prompt_version,firms,calls_logged,re_asked_firms,trimmed_firms,gpu,gpu_minutes_logged,last_session_finished
0,all_years,summary_lines_of_business,item1-mapreduce-1.3,1058,1070,2,0,NVIDIA A40,90.5,2026-10-26
1,three_years,summary_lines_of_business,item1-mapreduce-1.3,841,850,13,0,NVIDIA A40,27.5,2026-10-26
